In [ ]:
#pip install --upgrade pip
#pip install streamlit pandas langchain langchain-community langchain-google-genai google-generativeai chromadb langchain-ollama ollama python-dotenv
#ollama serve

In [ ]:
import streamlit as st
import pandas as pd
import os
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_core.documents import Document
from langchain.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

# ===================== RUTA FIJA DEL DATASET =====================
RUTA_DATASET = r"C:\Users\jelop\OneDrive\Documentos\U\2025-2\Pyoyecto_PLN_Local\DataSet\dataset_clasificado_PC\dataset_clasificado_PC.csv"

# ===================== CONFIG =====================
st.set_page_config(page_title="MediBot Local + Online", page_icon="Stethoscope", layout="wide")
st.title("MediBot – 6 Especialistas RAG (Local + Gemini)")

# ===================== MODO DE OPERACIÓN =====================
modo = st.sidebar.selectbox(
    "Modo de operación",
    ["Online (Gemini 2.5 Flash)", "Offline (Ollama Local)"],
    index=0
)

if modo == "Online (Gemini 2.5 Flash)":
    api_key = st.sidebar.text_input("Google API Key", type="password")
    if not api_key:
        st.warning("Ingresa tu API Key de Google para usar Gemini")
        st.stop()
else:
    st.sidebar.success("Modo Offline (Ollama) activado")

# ===================== CARGAR MODELOS =====================
@st.cache_resource
def cargar_modelos():
    if modo == "Online (Gemini 2.5 Flash)":
        from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0.3)
        embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004", google_api_key=api_key)
    else:
        from langchain_ollama import ChatOllama, OllamaEmbeddings
        llm = ChatOllama(model="llama3.1:8b", temperature=0.3)   # ← cámbialo si usas otro modelo
        embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")
    return llm, embeddings

llm, embeddings = cargar_modelos()

# ===================== CARGAR DATASET AUTOMÁTICO =====================
if not os.path.exists(RUTA_DATASET):
    st.error(f"No se encontró el dataset en:\n{RUTA_DATASET}\n\nAsegúrate de que la ruta sea correcta.")
    st.stop()

df = pd.read_csv(RUTA_DATASET)
if not {"Pregunta", "Respuesta", "Categoria"}.issubset(df.columns):
    st.error("El CSV debe tener las columnas: Pregunta, Respuesta, Categoria")
    st.stop()

st.success(f"Dataset cargado automáticamente: {len(df):,} casos")
st.sidebar.write(f"Total casos: **{len(df):,}**")
st.sidebar.dataframe(df["Categoria"].value_counts())

# ===================== CATEGORÍAS =====================
CATEGORIAS = [
    "Symptom/Diagnosis", "Treatment/Medication", "Prevention/Health Advice",
    "Test/Interpretation", "Emergency/Critical", "Post-surgery/Recovery"
]

# ===================== CREAR 6 RAGs =====================
@st.cache_resource(show_spinner="Creando los 6 especialistas médicos...")
def crear_rags(_df):
    progress = st.progress(0)
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    retrievers = {}

    for i, cat in enumerate(CATEGORIAS):
        subdf = _df[_df["Categoria"] == cat]
        if subdf.empty:
            progress.progress((i + 1) / len(CATEGORIAS))
            continue

        docs = [
            Document(page_content=f"Q: {r.Pregunta}\nA: {r.Respuesta}", metadata={"cat": cat})
            for _, r in subdf.iterrows() if pd.notna(r.Pregunta) and pd.notna(r.Respuesta)
        ]
        chunks = splitter.split_documents(docs)

        db = Chroma.from_documents(
            chunks, embeddings,
            collection_name=f"medi_{cat.lower().replace('/', '_').replace(' ', '_')}",
            persist_directory="./chroma_local"
        )
        retrievers[cat] = db.as_retriever(search_kwargs={"k": 6})
        progress.progress((i + 1) / len(CATEGORIAS))

    st.success("¡Los 6 especialistas médicos están listos!")
    return retrievers

retrievers = crear_rags(df)

# ===================== TRADUCIR Y CLASIFICAR =====================
def traducir(texto, a="es"):
    prompt = f"Translate to {'Spanish' if a=='es' else 'English'} (solo el texto):\n\n{texto}"
    try:
        return llm.invoke([HumanMessage(content=prompt)]).content.strip()
    except:
        return texto

def clasificar(pregunta_en):
    prompt = f"""Classify into EXACTLY ONE of these categories:
{', '.join(CATEGORIAS)}

Question: {pregunta_en}

Answer ONLY the category name:"""
    try:
        resp = llm.invoke([HumanMessage(content=prompt)]).content.strip()
        for c in CATEGORIAS:
            if c.lower() in resp.lower():
                return c
        return "Symptom/Diagnosis"
    except:
        return "Symptom/Diagnosis"

# ===================== CHAT =====================
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "assistant", 
                                  "content": f"¡Hola! Estoy funcionando en modo **{modo}**.\nPregúntame cualquier cosa sobre salud en español."}]

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("¿En qué puedo ayudarte hoy?"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Traduciendo..."):
            pregunta_en = traducir(prompt, "en")
        with st.spinner("Asignando especialista..."):
            categoria = clasificar(pregunta_en)
            st.info(f"Especialista asignado: **{categoria}**")

        if categoria == "Emergency/Critical":
            st.error("¡EMERGENCIA MÉDICA DETECTADA!\n\nPor favor, acude inmediatamente a urgencias o llama al número de emergencias (112 / 911).")
        else:
            with st.spinner("Consultando casos reales..."):
                qa = RetrievalQA.from_chain_type(
                    llm=llm,
                    chain_type="stuff",
                    retriever=retrievers[categoria],
                    return_source_documents=True,
                    chain_type_kwargs={"prompt": PromptTemplate(
                        template=f"You are a highly specialized doctor in {categoria}.\nAnswer empathetically and precisely in English, using only real similar cases.\n\nContext: {{context}}\nQuestion: {{question}}\nAnswer:",
                        input_variables=["context", "question"]
                    )}
                )
                result = qa({"query": pregunta_en})
                respuesta_en = result["result"]
                fuentes = result["source_documents"]

            respuesta_es = traducir(respuesta_en, "es")
            st.write(respuesta_es)

            with st.expander(f"Fuentes reales utilizadas ({len(fuentes)} casos similares)"):
                for i, doc in enumerate(fuentes[:3]):
                    st.caption(f"Caso {i+1}:\n{doc.page_content[:600]}...")

        st.session_state.messages.append({"role": "assistant", "content": respuesta_es})



# AÑADE ESTO AL FINAL DE TU NOTEBOOK (después de todo el código)
import os
codigo = open(__file__, encoding='utf-8').read() if '__file__' in globals() else In[-1]

with open("medi_bot_local.py", "w", encoding="utf-8") as f:
    f.write(codigo)

st.success("Archivo medi_bot_local.py creado correctamente en esta carpeta")
st.code("Ejecuta en terminal: streamlit run medi_bot_local.py")

In [ ]:
# CELDA FINAL: Lanzar la app
import subprocess
subprocess.Popen(["streamlit", "run", "medi_bot_local.py"])
print("¡MediBot iniciado! → Abre tu navegador en http://localhost:8501")